# HACK the WAVE: Leveraging Data Science to help fight ALS

## Track 3: Bioinformatics

**GOAL:** Develop a method to accurately predict the change in ALSFRS for an ALS patient over time.

**What is ALSFRS?** Amyotrophic Lateral Sclerosis Functional Rating Scale (ALSFRS) is an instrument for evaluating the functional status of patients with Amyotrophic Lateral Sclerosis. It can be used to monitor functional change in a patient over time. Specifically it measures:
- speech
- salivation
- swallowing
- handwriting
- cutting food and handling utensils (with or without gastrostomy)
- dressing and hygiene
- turning in bed and adjusting bed clothes
- walking
- climbing stairs
- breathing

In this notebook, we will explain show some examples of interacting with the data.

In [ ]:
import numpy as np
import pandas as pd
from os import listdir
from IPython.display import display

## Reading in the ProAct Data

In this challenge, we will be inspecting the ProAct dataset, which is comprised of 13 different data files. Each of these files include different metrics which may be useful for arriving at an accurate estimate of ALSFRS&mdash;determing which metrics are useful is for you to decide!

To understand the physical meaning behind each of these metrics, take a look at this website, https://nctu.partners.org/ProACT/Document/DisplayLatest/2

In [ ]:
!ls ../input/alsa/pro_act_dataset\ 2/Pro_Act_Dataset/

In [ ]:
data_dir = "../input/alsa/pro_act_dataset 2/Pro_Act_Dataset/"  # Specifying where I have the data stored

Here we will extract all the data from the ProAct data set and print some intermediate metrics to get an idea of the contents



In [ ]:
dfs = {}

# Loop over files in directory
for file in listdir(data_dir):
    
    if ".csv" not in file:
        continue
    
    # Extract file name without .csv extension
    file_ref =  file[0:-4].lower()  # casting string to lowercase
    print(f"Generating {file_ref} DataFrame...")
    
    _df = pd.read_csv(data_dir+file)
    
    # Summarize the contents of the dataframe
    display(_df.describe(include='all'))  # Using `display` to prettify output
    
    print(f"\t {file_ref} shape: {_df.shape}")
    
    # Storing each dataframe in a dictionary, for convinent storage
    dfs[file_ref] = _df
    
    # Delete the individual DataFrame to free up some memory
    del _df  

And we can verify that we have all the data properly stored in the `dfs` dictionary,

In [ ]:
print(dfs.keys())

dfs["treatment"].head()

## Comments on Data Cleaning

It doesn't matter which fancy machine learning algorithm you use, if you're data isn't reliable, neither are your results. This idea is best captured by the phrase,

> ...garbage in, garbage out...

Thus, we will need to start the painstaking process of cleaning the data.

As we can see below, in some data sets, many patients have more than one record. In this example, subject id 329 has many records (as they have undergone multiple tests).

In [ ]:
dfs["labs"][dfs["labs"]["subject_id"] == 329].head()

Additionally, for many features, a large portion of the data is missing, as seen in the example below.

In [ ]:
dfs["svc"].head()

**So your first task** is to clean the data and impute (or drop) features which won't aid in prediction accuracy.

## Calculating the ALSFRS Slope

If you were to inspect the alsfrs dataset, you may notice that a slope column is not available. This data set encodes the measurements over periods of time `ALSFRS_Delta`, so here is an explanation of how to compute this change over time:

> The algorithms developed for the challenge were required to predict the slope of change between months 3 to 12 after trial onset. In order to determine slopes, solvers were required to do the following:

> 1. To assure similarity when designing the prediction algorithms, solvers were asked for the predictions of slopes based on 10 questions (either 1-10 in ALSFRS or 1-9 +10a (dyspnea) in ALSFRS-R). For patients with ALSFRS scores, their ALSFRS Total sum [ALSFRS Total] should be used. For patients with ALSFRS-R scores, the total is generated using the sum of the following parameters: [ALSFRS-RSpeech, Salivation, Swallowing, Handwriting, Cutting (with and without Gastrostomy), Dressing and Hygiene, Turning in Bed, Walking, Climbing Stairs, Dyspnea] (the results of questions 10b and 10c are discarded when calculating the sum). In both cases, the number should range between 0–40.
> 2. Merge ALSFRS questions 5a. Cutting without Gastrostomy and 5b. Cutting with Gastrostomy
> 3. Remove all ALSFRS values for the time points in which NOT all 10 ALSFRS questions are available
> 4. Convert days to months: m= (days/365.24) *12)
> 5. Slopes between months 3 and 12 are then calculated as:
(ALSFRS (LastVisit)-ALSFRS(FirstVisit)) / (months(LastVisit-FirstVisit))
> 
> Definitions:
>
> **First Visit:** Assign first visit > 3 months (= 92 days) from the first time ALSFRS was fully measured (Reference Visit) as "First Visit" (this is the first visit after 3 month)
Note: for the calculation, set the first visit with 10 ALSFRS questions as the Reference Visit for slope calculation and hence calculate all differences relative to this visit. Note that the Reference Visit is not necessarily at delta=0.
> 
> **Last Visit:** If there are multiple visits > 12th month, assign the earliest visit > 12th month (from the Reference Visit for slope calculation) as ‘Last Visit’.
Otherwise: use final assessment of ALSFRS as “Last Visit”.

For more information, see this page: https://nctu.partners.org/ProACT/Document/DisplayLatest/3

Below are some example pieces of code which may help with this process, and show some Pandas functionality&mdash;for those unfamiliar.

In [ ]:
alsfrs_copy = dfs["alsfrs_training"]
alsfrs_copy.head()

### Example Merge Questions 5a and 5b

In this example, I filled all `NaN` values with zero using the `.fillna()` function, then used the `.sum()` function to sum the two columns. However, feel free to try an alternative method&mdash;perhaps you would rather impute, rather than drop the value to 0.

In [ ]:
# Create a new column called "Q5_Merge", where we will store the summed values
alsfrs_copy["Q5_Merge"] = alsfrs_copy[["Q5a_Cutting_without_Gastrostomy", "Q5b_Cutting_with_Gastrostomy"]].fillna(0).sum(axis=1)

alsfrs_copy["Q5_Merge"].head()

### Example Convert to Months

The progression of time is noted in the `ALSFRS_Delta` column, and is units of days. Here is a quick way to convert the days to months using `.apply()` and `lambda`. If you prefer to use a library, `astropy.units` is a good choice, https://docs.astropy.org/en/stable/units/#module-astropy.units.

In [ ]:
# For each row in the "ALSFRS_Delta", divide by 365.24 and multiply by 12
alsfrs_copy["ALSFRS_Delta"] = alsfrs_copy["ALSFRS_Delta"].apply(lambda x:  (x/365.24) * 12)

### Example to Help with Slope Calculation

Here is an example of how to use Pandas `.groupby` to select all the records related to each patient, and use the `.last()` and `.first()` to compute a difference.

In [ ]:
_temp = alsfrs_copy[["subject_id", "ALSFRS_Total", "ALSFRS_R_Total", "ALSFRS_Delta"]].groupby("subject_id")

_temp = _temp.last() - _temp.first()

_temp.head()

From here you can figure out how to calculate the slope using the information provided above. **Warning:** this is **not** a complete example of calculating the slope, as a few steps were skipped. You must go through each step discussed above, or else this will dramatically impact your accuracy upon submission.

## Formatting Your Ouput

As stated before, the goal of this track is to generate an accurate ALSFRS slope prediction. The format of your submission should be in the format of a **.csv** with an `Id` column identifying the row and a `Slope` column specifying the true value which you are attempting to predict.

| Id| Slope |
| :---: | :----:|
| ... | ... |
| ... | ... |
| ... | ... |

<br>
You will be score on the Root mean squared error (RMSE). You can resubmit a **maximum 20 times a day**.

**WARNING:** The submisssion system is **not** currently active. It will be activated tomorrow (9/21/2019).
